# 08 · Capstone: ShipFast and the research agent in CrewAI

Notebooks 01 to 07 built the pieces. This notebook puts them together into the two agents every
framework in this project implements against the same contract (`contracts/`), and shows how they are
evaluated. The agents live in `crew_agents/`, and `run_eval.py` runs the full evaluation with the same code.

**You will see**

- the ShipFast agent's design in CrewAI, and why its Resolution is built in code
- two hard ShipFast scenarios run live through the **shared runner**: an approval, and a two-turn case
- one research question through the shared research runner
- this framework's full results, and all three frameworks side by side

In [1]:
import crew_agents.shipfast as sf  # import crew_agents first: telemetry and panels off
from crew_agents.research_flow import answer
from crew_agents.shipfast import CrewAIShipFast

from agent_contracts.harness import verify_model
from agent_contracts.harness.runners import run_question, run_scenario
from agent_contracts.research.scoring import load_questions
from agent_contracts.shipfast.scoring import load_scenarios

verify_model()
scenarios = {s["id"]: s for s in load_scenarios()}
questions = {q["id"]: q for q in load_questions()}

## The ShipFast agent

In CrewAI the agent is a **Flow**, one run per customer turn: triage, a router, a fixed decline or the specialist agent with
the gated refund, and a `@human_feedback` step that pauses with the state saved by `@persist`. The
conversation history travels in the flow's state from turn to turn.

Every build fills in the scored `Resolution` the same way, with the shared
`agent_contracts.shipfast.resolution.derive_resolution` (DECISIONS.md 16): `intent` and `order_id` from
triage; `action`, `refund_amount` and `ticket_id` from what actually happened in the ledger and the
tickets; and only when a turn left no record does the model choose between `answered`, `needs_info` and
`declined`. The component notebooks showed why: asked to fill in a Resolution itself, the 8B model
claimed refunds that never happened and invented ticket numbers.

In [2]:
print(sf.__doc__)

The ShipFast support agent in CrewAI (ShipFast SPEC.md), the capstone of notebooks 01 to 07.

One Flow run per customer turn; the conversation history travels in the flow's state from turn to turn.

    triage (router) --"out_of_scope"--> send_decline
                    \--"in_scope"----> run_specialist (router) --"finished"--> done
                                                               \--"needs_approval"--> ask_supervisor
                                                                    (@human_feedback: pauses, state saved)
                                                                    --"approve"--> issue_approved_refund (code)
                                                                    --"reject"---> escalate_rejected_refund (agent)

The approval gate works as in notebook 06 (crew_agents/approval.py): the specialist's `issue_refund` is
gated above the shared $100 limit, the flow pauses between steps, and it resumes only from saved state.
The Resolution is b

In [3]:
print([m for m in ("triage", "route", "send_decline", "run_specialist", "check_refund", "ask_supervisor",
                  "issue_approved_refund", "escalate_rejected_refund") if hasattr(sf.ShipFastFlow, m)])

['triage', 'route', 'send_decline', 'run_specialist', 'check_refund', 'ask_supervisor', 'issue_approved_refund', 'escalate_rejected_refund']


## Run through the shared runner

`run_scenario` is the same function every framework's `run_eval.py` uses. It resets the shared tools,
feeds each turn, resumes a paused approval with the scenario's decision, and checks the result against
`scenarios.json`. The framework only supplies the adapter: start a conversation, send a turn, resume.

In [4]:
def show(item):
    print(f"{item['id']}: {'PASS' if item['passed'] else 'FAIL'}  ({item['latency_s']:.0f}s, "
          f"{item['llm_calls']} model calls, tools {item['tool_calls']})")
    for i, t in enumerate(item["turn_resolutions"], 1):
        print(f"  turn {i}: {t['intent']} {t['order_id']} -> {t['action']} {t['refund_amount'] or ''}")
        print(f"          {t['reply'].strip()[:160]}")
    for p in item["problems"]:
        print("  problem:", p)


show(await run_scenario(CrewAIShipFast(), scenarios["S3a_late_refund_approved"]))

╭──────────────────────────────────────────────── ⏳ Flow Paused ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Paused - Waiting for Feedback                                                                             │
│  Name: ShipFastFlow                                                                                             │
│  ID: b63c0e9b-c979-496b-a2db-0af2482695e8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

S3a_late_refund_approved: PASS  (49s, 7 model calls, tools ['lookup_order', 'track_shipment', 'search_policy', 'issue_refund'])
  turn 1: refund_request A1003 -> refunded 250.0
          Good news: your refund of $250.00 for order A1003 has been approved and issued.


S3a is the late espresso machine, a $250 refund the supervisor approves. A passing run pauses at the
approval gate, is resumed with `approve`, and issues exactly $250.

In [5]:
show(await run_scenario(CrewAIShipFast(), scenarios["S4_damaged_needs_photo_then_refund"]))

S4_damaged_needs_photo_then_refund: FAIL  (78s, 12 model calls, tools ['lookup_order', 'track_shipment', 'search_policy', 'issue_refund', 'lookup_order', 'search_policy', 'issue_refund'])
  turn 1: complaint A1004 -> refunded 60.0
          The refund for your order A1004 has been issued. Please provide a photo of the damaged vase set so we can process the refund. Thank you for your understanding.
  turn 2: complaint A1004 -> answered 
          The refund for your order A1004 has already been processed in full. If you need further assistance, please let me know. Thank you for your understanding.
  problem: action: expected 'refunded', got 'answered'
  problem: refund_amount: expected 60.0, got None
  problem: turn 1: action: expected 'needs_info', got 'refunded'
  problem: turn 1: refund_amount: expected None, got 60.0
  problem: turn 2: action: expected 'refunded', got 'answered'
  problem: turn 2: refund_amount: expected 60.0, got None


S4 is the broken vase: the first turn must ask for a photo and refund nothing (policy P3); the second,
which never names the order, must refund the $60 total. It needs memory and policy in the right order.

When we ran them, S3a passed: the gated refund recorded $250 as pending, the flow paused at
`@human_feedback`, was restored from saved state and resumed with `approve`, and code issued exactly $250.
S4 failed the way it did in notebooks 03 and 04: the specialist refunded $60 on the first turn, before
any photo, while telling the customer to send one. On the second turn the shared `issue_refund` refused a
second refund (DECISIONS.md 15), so the ledger holds one $60, but the money moved before policy P3 allowed
it, and the scenario fails.

## One research question

The research agent is the workflow from notebook 05, answered through the shared `run_question`. Q07 has
a stale article (9 provinces) beside the current one (14 provinces); mentioning the stale figure at all
fails the answer.

In [6]:
item = await run_question(answer, questions["Q07_deployment"])
print("passed:", item["passed"], "| tools:", item["tool_calls"], "| model calls:", item["llm_calls"], f"| {item['latency_s']:.0f}s")
print("answer:", item["report"]["answer"])
print("problems:", item["problems"] or "none")

passed: True | tools: ['web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search'] | model calls: 10 | 171s
answer: Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces, according to the 2025 annual report [F04].
problems: none


## The full evaluation

`run_eval.py` runs every scenario and every question once and writes `results/shipfast/crewai.json` and
`results/research/crewai.json` (from the tutorial folder: `uv run python run_eval.py`). It takes about half an
hour on a 16 GB M4 Mac, so here we read the saved results instead of re-running them.

In [7]:
import json
from pathlib import Path

RESULTS = Path("..").resolve().parent / "results"


def load(agent, framework):
    path = RESULTS / agent / f"{framework}.json"
    return json.loads(path.read_text()) if path.exists() else None


for agent, key in (("shipfast", "scenarios"), ("research", "questions")):
    data = load(agent, "crewai")
    print(f"== {agent}: {data['summary']['passed']} of {data['summary']['total']} passed (run at {data['run_at']})")
    for it in data[key]:
        if not it["passed"]:
            print(f"  {it['id']}: {it['problems']}")

== shipfast: 7 of 8 passed (run at 2026-11-03T08:34:31+00:00)
  S4_damaged_needs_photo_then_refund: ["action: expected 'refunded', got 'answered'", 'refund_amount: expected 60.0, got None', "turn 1: action: expected 'needs_info', got 'refunded'", 'turn 1: refund_amount: expected None, got 60.0', "turn 2: action: expected 'refunded', got 'answered'", 'turn 2: refund_amount: expected 60.0, got None']
== research: 9 of 10 passed (run at 2026-11-03T08:55:05+00:00)
  Q10_order_cost: ["missing facts: ['price_40', 'price_mini']"]


## All three frameworks

The same contract, data, model, settings, tools, instructions, scoring and runner, so the differences
below come from the frameworks (and from how a small model reacts to each one's way of packaging the
same request).

In [8]:
rows = []
for agent in ("shipfast", "research"):
    for framework in ("langgraph", "crewai", "strands"):
        data = load(agent, framework)
        if data:
            s = data["summary"]
            rows.append((agent, framework, f"{s['passed']}/{s['total']}", s["total_tokens"], round(s["total_latency_s"])))
print(f"{'agent':9} {'framework':10} {'passed':7} {'tokens':>7} {'seconds':>8}")
for r in rows:
    print(f"{r[0]:9} {r[1]:10} {r[2]:7} {r[3]:7} {r[4]:8}")

agent     framework  passed   tokens  seconds
shipfast  langgraph  7/8       29962      387
shipfast  crewai     7/8       34867      401
shipfast  strands    4/8       26981      331
research  langgraph  10/10     44361     1464
research  crewai     9/10      60243     1194
research  strands    9/10      48803     1147


CrewAI passed 7 of 8 ShipFast scenarios and 9 of 10 research questions. The ShipFast failure is S4 above.
The research failure, Q10, gave the correct total (PHP 1,635,000) and both subtotals but not the two unit
prices, which the question's expected facts include; whether the check should require them is an open
question for the contract.

Across the frameworks, CrewAI ties LangGraph on ShipFast with a different single failure, and used the
most tokens on both agents: each CrewAI agent call carries a longer prompt (role, goal, backstory), and its
research agent searched more often. These are single runs on one laptop: a first measurement, not a
ranking.

## How the others do this

- **LangGraph** builds the same agent as a `StateGraph` with the specialist as a subgraph; the approval
  pause happens inside the tool call, and a checkpointer holds the conversation and the pause.
- **Strands** uses **one agent per conversation**, with triage as a separate structured call and the
  approval as a hook that interrupts before the refund.

## Takeaways

- The contract keeps the comparison fair: one copy of the data, tools, instructions, scoring and runner.
- Code records the facts (refunds, tickets, pauses); the model writes the reply and makes one small call.
- Every framework could build both agents, but each needed its own shape for the approval pause and for
  memory, and each made different mistakes with the same model. Those differences are the comparison.